## Convert raw CSV to HTTP request-level DataFrame

> **Purpose:**
> Perform an exploratory analysis of the raw CSIC 2010 HTTP traffic dataset, at the column and value level, to understand how the data is represented.
>
> **Goals:**
> 1. Examine the structure of the raw CSV: columns, value distributions, missing/empty fields, and data types.
> 2. Deduce how CSV rows and columns map to actual HTTP requests (method, URL, headers, payload), and reconstruct the request-level representation.
> 3. Store the resulting preprocessed dataset for use in feature engineering (`featurize.ipynb`).
>
> **Output:** `data/csic_2010_preprocessed.csv/.parquet`

---

**Exploratory analysis of the raw CSIC 2010 dataset.**
> Understand the raw CSV at the column and value level, deduce the mapping between CSV records and HTTP request structure, and store a preprocessed dataset ready for feature engineering.
---


In [1]:
import os
from pathlib import Path

import pandas as pd


In [44]:
path = Path.cwd()
data_path = path.parent / "data"
input_path = data_path / "csic_2010_raw.csv"
output_path = data_path / "csic_2010_preprocessed.csv" 

### load the dataset

In [45]:
df_raw = pd.read_csv(input_path, low_memory=False)
df_raw

,index,method,url,protocol,userAgent,pragma,cacheControl,accept,acceptEncoding,acceptCharset,acceptLanguage,host,connection,contentLength,contentType,cookie,payload,label
0,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,id=2,anom
1,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,nombre=Jam%F3n+Ib%E9rico,anom
2,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,precio=85,anom
3,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,cantidad=%27%3B+DROP+TABLE+usuarios%3B+SELECT+...,anom
4,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,B1=A%F1adir+al+carrito,anom
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223580,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,238.0,application/x-www-form-urlencoded,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,ciudad=Hontangas,norm
223581,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,238.0,application/x-www-form-urlencoded,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,cp=08192,norm
223582,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,238.0,application/x-www-form-urlencoded,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,provincia=Teruel,norm
223583,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,238.0,application/x-www-form-urlencoded,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,ntc=8231060336250168,norm


**Observations:**  

The raw CSV contains:

- The raw dataset contains approximately 223,585 parameter level rows and 18 columns.
- These rows may represent logical HTTP requests grouped by `index`.  
- The `method` represents HTTP methods, such as GET, POST, or PUT. Can be used a categorial column
- The `url` column Request URL or URI. Can be featurised in multiple ways based on:
  - Counts on special chars in url such as '&', '/', '.', '=' etc
  - Count of query parameters
  - Length of `url`
  - Presence of well know suspicious parterns such as:
    - sql: select, union, drop, insert, delete
    - html: script tag
    - Path traversal pattern
- The `protocol` column represents HTTP protocol version. Seems to be a constant value column
- The `user_agent`column represents User-Agent header value. Seems to be a constant value column
- HTTP header field columns `pragma`, `cache_control`, `accept`, `accept_encoding`, `accept_charset`, `accept_language`, `host` and `connection` seem to be constant value columns  
- The `content_length` contains a numeric content length value
- The `contentType`  can be used a categorial column
- The `payload` columns represent the request payload. Can be featurised in multiple ways based on: 
    - Whether payload is present or not
    - Length of `payload`
    - Counts on special chars in payload such as '&', '/', '.', '=' etc
    - Presence of well know suspicious parterns such as:
        - sql: select, union, drop, insert, delete
        - html: script tag
        - Path traversal pattern
- The `label` is the target column with `anom` (anomalous) and `norm` (normal) values

### Check the data types and missing values in the various columns:

In [4]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 223585 entries, 0 to 223584
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   index           223585 non-null  int64  
 1   method          223585 non-null  object 
 2   url             223585 non-null  object 
 3   protocol        223585 non-null  object 
 4   userAgent       223585 non-null  object 
 5   pragma          223585 non-null  object 
 6   cacheControl    223585 non-null  object 
 7   accept          223585 non-null  object 
 8   acceptEncoding  223585 non-null  object 
 9   acceptCharset   223585 non-null  object 
 10  acceptLanguage  223585 non-null  object 
 11  host            223585 non-null  object 
 12  connection      223585 non-null  object 
 13  contentLength   100135 non-null  float64
 14  contentType     100135 non-null  object 
 15  cookie          223585 non-null  object 
 16  payload         198094 non-null  object 
 17  label     

**Observations:**  
- `contentLength`, `contentType`, `payload` colums have values missing.
- The target `label` has no values missing i.e no rows to discard

### Analyse the uniqueness of each column

In [5]:
print(f"index: {df_raw['index'].nunique()}")
print(f"method: {df_raw['method'].nunique()}")
print(f"url: {df_raw['url'].nunique()}")
print(f"protocol: {df_raw['protocol'].nunique()}")
print(f"userAgent: {df_raw['userAgent'].nunique()}")
print(f"pragma: {df_raw['pragma'].nunique()}")
print(f"cacheControl: {df_raw['cacheControl'].nunique()}")
print(f"accept: {df_raw['accept'].nunique()}")
print(f"acceptEncoding: {df_raw['acceptEncoding'].nunique()}")
print(f"acceptCharset: {df_raw['acceptCharset'].nunique()}")
print(f"acceptLanguage: {df_raw['acceptLanguage'].nunique()}")
print(f"host: {df_raw['host'].nunique()}")
print(f"connection: {df_raw['connection'].nunique()}")
print(f"contentLength: {df_raw['contentLength'].nunique()}")
print(f"contentType: {df_raw['contentType'].nunique()}")
print(f"cookie: {df_raw['cookie'].nunique()}")
print(f"payload: {df_raw['payload'].nunique()}")
print(f"label: {df_raw['label'].nunique()}")

index: 36000
method: 3
url: 1643
protocol: 1
userAgent: 1
pragma: 1
cacheControl: 1
accept: 1
acceptEncoding: 1
acceptCharset: 1
acceptLanguage: 1
host: 2
connection: 1
contentLength: 382
contentType: 1
cookie: 61065
payload: 33769
label: 2


**Observations:**  
- `index` column shows `36000` unique values whereas total rows count is `223585`, It implies that index column values are not unique. 
- `method` column with `3` unique values (GET, POST, PUT), Can be used as Categorial feature (one-hot encode)
- `url` with `1643` unique values. Various features columns can be derived from iteg: url_length, count of special characters, presense of malicious patterns etc
- `host` column has `2` unique values. Can be used as Categorical feature (one-hot encode)
- `contentLength` column has `282` unique values. Can be used as Numeric feature  column.
- `cookie` column has `61065` unique values. Can be used as Numeric feature  column.
- `payload` column with `33769` unique values. Various features columns can be derived from it eg: payload_length, count of special characters, presense of malicious patterns etc
- `protocol`, `userAgent`, `pragma`, `cacheControl`, `accept`, `acceptEncoding`, `acceptCharset`, `acceptLanguage`, `connection`, `contentType` all these columns have a single unique value. These columns can be dropped.
- `label` column has `2` values. Can be used as Categorical feature 

### Drop Constant Columns:

In [6]:
const_cols = ["protocol", "userAgent", "pragma", "cacheControl", "accept", "acceptEncoding", "acceptCharset", "acceptLanguage", "connection", "contentType"]

df_multi = df_raw.drop(columns = const_cols)
df_multi

,index,method,url,host,contentLength,cookie,payload,label
0,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,id=2,anom
1,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,nombre=Jam%F3n+Ib%E9rico,anom
2,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,precio=85,anom
3,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,cantidad=%27%3B+DROP+TABLE+usuarios%3B+SELECT+...,anom
4,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,B1=A%F1adir+al+carrito,anom
...,...,...,...,...,...,...,...,...
223580,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,localhost:8080,238.0,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,ciudad=Hontangas,norm
223581,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,localhost:8080,238.0,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,cp=08192,norm
223582,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,localhost:8080,238.0,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,provincia=Teruel,norm
223583,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,localhost:8080,238.0,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,ntc=8231060336250168,norm


### Analyze rows for `index=0`

In [7]:
group_index0 = df_multi[df_raw['index'] == 0]
group_index0

,index,method,url,host,contentLength,cookie,payload,label
0,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,id=2,anom
1,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,nombre=Jam%F3n+Ib%E9rico,anom
2,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,precio=85,anom
3,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,cantidad=%27%3B+DROP+TABLE+usuarios%3B+SELECT+...,anom
4,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,B1=A%F1adir+al+carrito,anom
119585,0,GET,http://localhost:8080/tienda1/index.jsp,localhost:8080,NaN,JSESSIONID=1F767F17239C9B670A39E9B10C3825F4,NaN,norm


**Observations:**  

- HTTP Request 1 (Anomalous):  
  - Rows: `0` to `0`  
  - Cookie: `JSESSIONID=B92A8B48B9008CD29F622A994E0F650D`  
  - Label: `anom`  
  - Payload: Fragmented `id=2`, `nombre=Jam%F3n+Ib%E9rico`, `precio=85`, `cantidad=%27%3B+DROP+TABLE+usuarios%3B+SELECT+*+FROM+datos+WHERE+nombre+LIKE+%27%25`, `B1=A%F1adir+al+carrito`
- HTTP Request 2 (Normal):  
  - Rows: 119585  
  - Cookie: `JSESSIONID=1F767F17239C9B670A39E9B10C3825F4`  
  - Label: `norm`  
  - Payload: `NaN`  
- `index=0` is reused for both requests.  
- `cookie` uniquely distinguishes both requests.  
- `Payload` is split across multiple rows for the for HTTP Request 1. Combine fragments to reconstruct the full payload.  
- All rows in a HTTP request share the same `label` (`anom` or `norm`).  

### Analyze rows for `index=1`

In [8]:
group_index1 = df_multi[df_raw['index'] == 1]
group_index1

,index,method,url,host,contentLength,cookie,payload,label
5,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=F563B5262843F12ECAE41815ABDEEA54,id=2%2F,anom
6,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=F563B5262843F12ECAE41815ABDEEA54,nombre=Jam%F3n+Ib%E9rico,anom
7,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=F563B5262843F12ECAE41815ABDEEA54,precio=85,anom
8,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=F563B5262843F12ECAE41815ABDEEA54,cantidad=49,anom
9,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=F563B5262843F12ECAE41815ABDEEA54,B1=A%F1adir+al+carrito,anom
119586,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,id=3,norm
119587,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,nombre=Vino+Rioja,norm
119588,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,precio=100,norm
119589,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,cantidad=55,norm
119590,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,B1=A%F1adir+al+carrito,norm


**Observations:**  

- HTTP Request 1 (Anomalous):  
  - Rows: `5` to `9`  
  - Cookie: `JSESSIONID=F563B5262843F12ECAE41815ABDEEA54`  
  - Label: `anom`  
  - Payload: Fragmented `id=2%2F`, `nombre=Jamón`, `precio=85`, `cantidad=49`, `B1=A%F1adir+al+carrito`
- HTTP Request 2 (Normal):  
  - Rows: 119586–119590  
  - Cookie: `JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5`  
  - Label: `norm`  
  - Payload: Fragmented `id=3`, `nombre=Vino+Rioja`, `precio=100`, `cantidad=55`, `B1=A%F1adir+al+carrito`  
- `index=1` is reused for both requests.  
- `cookie` uniquely distinguishes both requests.  
- `Payloads` are split across multiple rows for the same request. Combine fragments to reconstruct the full payload.  
- All rows in a HTTP request share the same `label` (`anom` or `norm`).  

> **Observations:**  
> - `10` rows in total for `index=1`
> - Rows# `5` to `9` and `119586` to `119590` implies non-contiguous row distribution for `index=1`

**Based on the analysis of `index=0` and `index=1` following conclusions can be derived:**  
1. `cookie` is a Unique Identifier for HTTP Requests  
2. `index` column is not unique
3. `Payload` column fragments can be reassembled using cookie
4. All Rows in a HTTP Request share the same `label`

### Group the dataframe by `cookie` and inspect for inconsistent and missing values  

In [24]:
# Group the DataFrame by 'cookie'
grouped_by_cookie = df_multi.groupby('cookie')
print(f"Total groups: {len(grouped_by_cookie)}")

Total groups: 61065


There are `61065` HTTP requests  

In [32]:
from collections import defaultdict

# Initialize dictionaries to store counts
inconsistency_counts = defaultdict(int)
empty_string_counts = defaultdict(int)
nan_or_none_counts = defaultdict(int)

# Columns to check
columns_to_check = ['index', 'method', 'url', 'host', 'label', 'contentLength', 'payload']

# Iterate through each group and column
for name, group in grouped_by_cookie:
    for col in columns_to_check:
        # Check for inconsistencies (multiple unique values)
        unique_values = group[col].unique()
        if len(unique_values) != 1:
            inconsistency_counts[col] += 1

        # Check for empty strings ('')
        empty_string_mask = group[col] == ''
        if empty_string_mask.any():
            empty_string_counts[col] += 1

        # Check for NaN or None
        nan_or_none_mask = group[col].isna()
        if nan_or_none_mask.any():
            nan_or_none_counts[col] += 1


# Print the distribution for each column
for col in columns_to_check:
    print(f"\n{col}:")
    print(f"   - Inconsistent groups: {inconsistency_counts[col]}")
    print(f"   - Groups with empty strings: {empty_string_counts[col]}")
    print(f"   - Groups with NaN/None: {nan_or_none_counts[col]}")
    #print(f"   - Total groups: {len(grouped_by_cookie)}")
#print(f"\nTotal HTTP requests with no payload: {http_req_with_no_payload}")


index:
   - Inconsistent groups: 0
   - Groups with empty strings: 0
   - Groups with NaN/None: 0

method:
   - Inconsistent groups: 0
   - Groups with empty strings: 0
   - Groups with NaN/None: 0

url:
   - Inconsistent groups: 0
   - Groups with empty strings: 0
   - Groups with NaN/None: 0

host:
   - Inconsistent groups: 0
   - Groups with empty strings: 0
   - Groups with NaN/None: 0

label:
   - Inconsistent groups: 0
   - Groups with empty strings: 0
   - Groups with NaN/None: 0

contentLength:
   - Inconsistent groups: 0
   - Groups with empty strings: 0
   - Groups with NaN/None: 43088

payload:
   - Inconsistent groups: 23881
   - Groups with empty strings: 0
   - Groups with NaN/None: 25491


**Observtions:**  

- Values of `index`, `method`, `url`, `host` and `label` columns are consistent for the same HTTP request.
- `contentLength` for 43,088 HTTP requests are `NaN/None` which is expected for GET requests as they typically have no payload. It is primarily used for POST requests with payloads.
- `payload`
  - `Inconsistent groups: 23881` indicate payloads which are fragmented across rows for the same HTTP request. Each row in a group may have a different fragment of the payload.  
  - `Groups with NaN/None: 25491` indicate HTTP requests with no payload. This is expected for GET requests or other requests that do not include a payload.

> *Enhancement:* *To gain more insight, add more buckets per column to trace the sum up to `61065`*   
> *eg:*
>    - Non zero length and not fragmented payloads viz HTTP Request (`cookie`) spanning a signle row with non zero payload*
>    - Count of GET, POST, PUT in methods

### Reconstruct HTTP requests using `cookie` as the grouping key

In [34]:
df_reconstructed = (
    grouped_by_cookie.agg({
        'index': 'first',  # Take the first index (all rows in the group share the same index)
        'method': 'first',  # Take the first method
        'url': 'first',     # Take the first URL
        'host': 'first',     # Take the first host
        'label': 'first',   # Take the first label (all rows in the group share the same label)
        'payload': lambda x: ' '.join(x.dropna()),  # Combine all non-NaN payload fragments
        'contentLength': 'first',
    })
    .reset_index()  # Convert 'cookie' from index to a column
)

df_reconstructed

,cookie,index,method,url,host,label,payload,contentLength
0,JSESSIONID=0000442E26C2F96F52C97120FF00254B,6883,GET,http://localhost:8080/tienda1/imagenes/2.gif,localhost:8080,norm,,NaN
1,JSESSIONID=0000AF0F87A92C3DFEE827BDFC243736,4687,GET,http://localhost:8080/tienda1/publico/vaciar.j...,localhost:8080,anom,,NaN
2,JSESSIONID=00020FC2706BF9F82BE6D4A7BBE12F41,25623,GET,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,norm,id=2,NaN
3,JSESSIONID=0002AB6A7AC61AE9DC71925807460D8C,10367,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,anom,id=3 nombre=Jam%F3n+Ib%E9rico precio=%2500 can...,NaN
4,JSESSIONID=0003D3946ADEBD8EE20F3B4700193661,19522,POST,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,anom,id=2sessionid%3D12312312%26+username%3D%3Cscri...,174.0
...,...,...,...,...,...,...,...,...
61060,JSESSIONID=FFFE094F84AF5FA4FF65367134F7FA9A,35211,POST,http://localhost:8080/tienda1/publico/entrar.jsp,localhost:8080,norm,errorMsg=Credenciales+incorrectas,33.0
61061,JSESSIONID=FFFE2D706D5003EFE6113C6B721FDD97,18514,POST,http://localhost:8080/tienda1/miembros/editar.jsp,localhost:8080,anom,modo=registro login=fabris password=dEsEnquieT...,248.0
61062,JSESSIONID=FFFEC3CFA8BEA96ADBAB57FABFA311C2,5875,GET,http://localhost:8080/tienda1/publico/autentic...,localhost:8080,anom,modo=entrar login=quoi pwdA=gAMar9a remember=o...,NaN
61063,JSESSIONID=FFFF432CCE806C813B7A1CA89B874517,31132,POST,http://localhost:8080/tienda1/publico/pagar.jsp,localhost:8080,norm,modo=insertar precio=4760 B1=Confirmar,38.0


`df_reconstructed` DataFrame contains `61,065` rows (**reconstructed HTTP requests**) with `8` columns, and its structure is well-suited for further analysis

In [35]:
# TBD: Sample random `payload` to check reassembled ??

### Inspect the structure of reconstructed dataframe

In [36]:
df_reconstructed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   cookie         61065 non-null  object 
 1   index          61065 non-null  int64  
 2   method         61065 non-null  object 
 3   url            61065 non-null  object 
 4   host           61065 non-null  object 
 5   label          61065 non-null  object 
 6   payload        61065 non-null  object 
 7   contentLength  17977 non-null  float64
dtypes: float64(1), int64(1), object(6)
memory usage: 3.7+ MB


**Observations:**  
- The core metadata columns `cookie`, `index`, `method`, `url`, `host`, and `label`, `payload` are fully populated (`61,065` non-null entries). This confirms that each reconstructed HTTP request retains complete metadata. The grouping by `cookie` preserves the integrity of the original requests.  
- The `17,977` non-null values for `contentLength` align with the pre-reconstruction observation of `43,088` missing values *(61,065 total - 17,977 non-null = 43,088 missing)*.  

### Save the reconstructed dataframe to disk

In [51]:
# # Check for empty strings in the payload column
# empty_string_count = (df_reconstructed['payload'] == '').sum()
# print(f"Number of empty strings in 'payload': {empty_string_count}")

# # Check for NaN values
# nan_count = df_reconstructed['payload'].isna().sum()
# print(f"Number of NaN values in 'payload': {nan_count}")

# # Total non-null entries (including empty strings)
# non_null_count = df_reconstructed['payload'].notna().sum()
# print(f"Total non-null entries (including empty strings): {non_null_count}")

In [52]:
# df_reconstructed.to_csv(output_path, index=False)
# Save the DataFrame with na_rep='' to preserve empty strings
df_reconstructed.to_csv(output_path, index=False, na_rep='')

In [56]:
# Save as Parquet
%time
df_reconstructed.to_parquet(output_path.with_suffix('.parquet'))

CPU times: user 14 μs, sys: 1 μs, total: 15 μs
Wall time: 174 μs
